# Multi-Agent DevOps Incident Analysis Suite — LangGraph + LLM Colab v2

This is the real implementation notebook for the hackathon direction.

It includes:

- Upload raw log ZIP/files
- Parse and normalize logs
- Extract deterministic evidence
- Run actual LangGraph nodes
- Use LLM agents when an API key is available
- Fall back to deterministic agents when no key is available
- Generate RCA, remediation, Slack preview, ticket preview, cookbook, reports
- Score outputs against hidden `ground_truth_eval_only`

## Recommended Usage

1. Upload `block1_quick_raw_mixed_logs_v2.zip`.
2. Run all cells.
3. Tune prompts/rules.
4. Upload `block2_full_raw_mixed_logs_v2.zip` for final scoring.

Architecture:

```text
Raw logs -> Parser -> Evidence Extractor -> Evidence Clustering
         -> LangGraph:
            Classifier -> Timeline -> RCA -> Remediation -> Critic -> Action Builder
         -> Report + Eval
```

In [ ]:
!pip -q install pandas numpy pydantic tqdm scikit-learn matplotlib langgraph langchain langchain-openai langchain-core

In [ ]:
import os, re, json, zipfile, shutil
from pathlib import Path
from datetime import datetime
from collections import Counter
from typing import TypedDict, Optional, Any

import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, END

try:
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    LANGCHAIN_READY = True
except Exception as e:
    LANGCHAIN_READY = False
    print("LangChain import issue:", e)

WORK_DIR = Path("/content/devops_incident_suite_v2")
UPLOAD_DIR = WORK_DIR / "uploaded"
EXTRACT_DIR = WORK_DIR / "extracted"
OUTPUT_DIR = WORK_DIR / "outputs"

for d in [UPLOAD_DIR, EXTRACT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 180)
print("Workspace:", WORK_DIR)

## LLM Configuration

The notebook runs in two modes.

### No-key mode
Uses deterministic fallback agents.

### LLM mode
Set these environment variables before running the graph:

```python
os.environ["OPENAI_API_KEY"] = "..."
os.environ["MODEL_NAME"] = "gpt-4o-mini"
```

For OpenRouter:

```python
os.environ["OPENAI_API_KEY"] = "..."
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
os.environ["MODEL_NAME"] = "openai/gpt-4o-mini"
```

In [ ]:
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
USE_LLM = bool(os.getenv("OPENAI_API_KEY")) and LANGCHAIN_READY

def get_llm(temperature=0.1):
    if not USE_LLM:
        return None
    kwargs = {"model": MODEL_NAME, "temperature": temperature}
    if OPENAI_BASE_URL:
        kwargs["base_url"] = OPENAI_BASE_URL
    return ChatOpenAI(**kwargs)

print("USE_LLM:", USE_LLM)
print("MODEL_NAME:", MODEL_NAME)
print("OPENAI_BASE_URL:", OPENAI_BASE_URL)

## Upload Logs

In [ ]:
from google.colab import files

uploaded = files.upload()
for filename, content in uploaded.items():
    target = UPLOAD_DIR / filename
    with open(target, "wb") as f:
        f.write(content)
    print("Uploaded:", target)

In [ ]:
def reset_extract_dir():
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def extract_uploads():
    reset_extract_dir()
    for p in UPLOAD_DIR.iterdir():
        if p.suffix.lower() == ".zip":
            print("Extracting:", p.name)
            with zipfile.ZipFile(p, "r") as z:
                z.extractall(EXTRACT_DIR / p.stem)
        else:
            shutil.copy2(p, EXTRACT_DIR / p.name)

def discover_logs(root: Path):
    exts = {".jsonl", ".log", ".txt"}
    files = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts and "ground_truth_eval_only" not in str(p):
            files.append(p)
    return sorted(files)

extract_uploads()
log_files = discover_logs(EXTRACT_DIR)

print(f"Discovered {len(log_files)} runtime log file(s)")
for p in log_files[:50]:
    print(" -", p.relative_to(EXTRACT_DIR))

## Schemas

In [ ]:
class ParsedLog(BaseModel):
    source_file: str
    line_no: int
    ts: Optional[str] = None
    level: Optional[str] = None
    service: Optional[str] = None
    host: Optional[str] = None
    pod: Optional[str] = None
    region: Optional[str] = None
    env: Optional[str] = None
    version: Optional[str] = None
    trace_id: Optional[str] = None
    span_id: Optional[str] = None
    logger: Optional[str] = None
    message: str
    duration_ms: Optional[float] = None
    endpoint: Optional[str] = None
    status: Optional[int] = None
    raw: Any

class EvidenceItem(BaseModel):
    source_file: str
    line_no: int
    ts: Optional[str] = None
    level: Optional[str] = None
    service: Optional[str] = None
    message: str
    trace_id: Optional[str] = None
    signals: list[str] = Field(default_factory=list)
    weight: float = 0.0

class EvidenceCluster(BaseModel):
    cluster_id: str
    source_file: str
    candidate_category: str
    evidence_count: int
    weighted_score: float
    error_count: int
    warn_count: int
    affected_services: list[str]
    first_seen: Optional[str] = None
    last_seen: Optional[str] = None
    evidence_samples: list[dict]

class IncidentResult(BaseModel):
    incident_id: str
    source_file: str
    category: str
    severity: str
    confidence: float
    affected_services: list[str]
    first_seen: Optional[str] = None
    last_seen: Optional[str] = None
    evidence: list[dict]
    timeline: list[dict]
    primary_root_cause: str
    alternative_causes: list[str]
    remediation: list[str]
    validation_checks: list[str]
    safety_notes: list[str]
    critic_findings: list[str]
    ticket_required: bool
    notification_required: bool
    slack_preview: str
    ticket_preview: dict
    cookbook: str

## Parser

In [ ]:
TIMESTAMP_PATTERNS = [
    r"(?P<ts>\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?Z?)",
    r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:,\d+)?)"
]
LEVEL_PATTERN = r"\b(?P<level>TRACE|DEBUG|INFO|WARN|WARNING|ERROR|FATAL|CRITICAL)\b"

def normalize_level(level):
    if not level:
        return None
    level = str(level).upper()
    return {"WARNING":"WARN", "CRITICAL":"FATAL"}.get(level, level)

def parse_json_line(obj, source_file, line_no):
    msg = obj.get("message") or obj.get("msg") or obj.get("log") or json.dumps(obj)
    return ParsedLog(
        source_file=source_file,
        line_no=line_no,
        ts=obj.get("ts") or obj.get("timestamp") or obj.get("@timestamp") or obj.get("time"),
        level=normalize_level(obj.get("level") or obj.get("severity")),
        service=obj.get("service") or obj.get("app") or obj.get("component"),
        host=obj.get("host") or obj.get("node"),
        pod=obj.get("pod") or obj.get("container"),
        region=obj.get("region"),
        env=obj.get("env") or obj.get("environment"),
        version=str(obj.get("version")) if obj.get("version") is not None else None,
        trace_id=obj.get("trace_id") or obj.get("traceId") or obj.get("trace"),
        span_id=obj.get("span_id") or obj.get("spanId"),
        logger=obj.get("logger"),
        message=str(msg),
        duration_ms=float(obj.get("duration_ms")) if obj.get("duration_ms") is not None else None,
        endpoint=obj.get("endpoint") or obj.get("path") or obj.get("uri"),
        status=int(obj.get("status")) if str(obj.get("status", "")).isdigit() else None,
        raw=obj
    )

def parse_text_line(line, source_file, line_no):
    ts = None
    for pat in TIMESTAMP_PATTERNS:
        m = re.search(pat, line)
        if m:
            ts = m.group("ts")
            break
    m = re.search(LEVEL_PATTERN, line, re.IGNORECASE)
    svc_match = re.search(r"(?:service|app|component)=([A-Za-z0-9_.-]+)", line)
    trace_match = re.search(r"(?:trace_id|traceId|trace)=([A-Za-z0-9_.:-]+)", line)
    return ParsedLog(
        source_file=source_file,
        line_no=line_no,
        ts=ts,
        level=normalize_level(m.group("level")) if m else None,
        service=svc_match.group(1) if svc_match else None,
        message=line.strip(),
        trace_id=trace_match.group(1) if trace_match else None,
        raw=line
    )

def parse_file(path: Path):
    rows = []
    rel = str(path.relative_to(EXTRACT_DIR))
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for idx, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                rows.append(parse_json_line(obj, rel, idx).model_dump() if isinstance(obj, dict) else parse_text_line(line, rel, idx).model_dump())
            except Exception:
                rows.append(parse_text_line(line, rel, idx).model_dump())
    return rows

parsed = []
for p in tqdm(log_files, desc="Parsing logs"):
    parsed.extend(parse_file(p))

logs_df = pd.DataFrame(parsed)
print("Parsed rows:", len(logs_df))
display(logs_df.head())

## Signal Extraction and Clustering

In [ ]:
SIGNAL_RULES = {
    "database": [r"connection.*not available|connection.*timed out|jdbc connection|remaining connection slots|too many clients|pg_stat_activity|deadlock|lock wait|slow query|sequential scan|work_mem|fsync|checkpoint"],
    "network": [r"connection reset|reset by peer|dns|name resolution|no such host|tls handshake|ssl handshake|certificate verify|upstream connect|tcp retransmits|connection refused"],
    "authentication": [r"jwt|token|issuer mismatch|signature|jwk|401|403|rbac|authorization|auth middleware|permission|entitlement"],
    "memory_cpu": [r"oomkilled|oom killed|heap|oldgen|gc pause|cpu throttling|cpu saturation|cfs_throttled|memory limit|back-off restarting|resource pressure"],
    "deployment_regression": [r"rollout|deployment|canary|new error signature|version 2\.8\.1|rollback|feature flag|configmap|configuration reloaded|checksum|new code path"],
    "api_timeout": [r"504|gateway timeout|request timed out|upstream timed out|proxy_read_timeout|p95 latency|client timeout|circuit breaker|downstream timeout|read timeout"],
    "queue_backlog": [r"consumer lag|records-lag|max.poll.interval|queue depth|backlog|retry topic|dead-letter|poison|rebalance|partition assignment|heartbeat failed"],
    "disk_storage": [r"no space left|disk pressure|errno=28|ephemeral storage|evicted|logrotate|filesystem free|disk full|iowait|persistent volume|write latency"],
    "external_dependency": [r"provider|vendor|third-party|external|rate limit|429|Retry-After|market data vendor|feed delayed|sequence gap|secondary feed"],
    "unknown": [r"unexpected state|unclassified|unknown|missing correlation|trace context.*absent|incomplete trace|manual inspection|insufficient evidence|redacted"],
}
SEVERITY_WEIGHTS = {"FATAL": 4.5, "ERROR": 3.0, "WARN": 1.5, "INFO": 0.2, "DEBUG": 0.05, None: 0.1}

def extract_signals(row):
    text = f"{row.get('message','')} {row.get('service','')} {row.get('logger','')} {row.get('status','')}".lower()
    hits = []
    for cat, patterns in SIGNAL_RULES.items():
        if any(re.search(pat, text, flags=re.IGNORECASE) for pat in patterns):
            hits.append(cat)
    return hits

logs_df["signals"] = logs_df.apply(extract_signals, axis=1)
logs_df["has_signal"] = logs_df["signals"].apply(bool)
logs_df["signal_weight"] = logs_df.apply(lambda r: SEVERITY_WEIGHTS.get(r.get("level"), 0.1) * max(1, len(r["signals"])), axis=1)

print("Rows with signals:", int(logs_df["has_signal"].sum()))
display(logs_df[logs_df["has_signal"]][["source_file","line_no","ts","level","service","message","signals"]].head(10))

def build_clusters(logs_df, max_samples=12):
    evidence_df = logs_df[logs_df["has_signal"]].copy()
    exploded = evidence_df.explode("signals").rename(columns={"signals": "candidate_category"})
    clusters = []
    for (source_file, cat), g in exploded.groupby(["source_file", "candidate_category"]):
        services = sorted(set([str(x) for x in g["service"].dropna() if str(x) and str(x) != "nan"]))
        sample_df = g.sort_values(["signal_weight"], ascending=False).head(max_samples)
        samples = []
        for _, r in sample_df.iterrows():
            samples.append(EvidenceItem(
                source_file=r["source_file"], line_no=int(r["line_no"]), ts=r.get("ts"), level=r.get("level"),
                service=r.get("service"), message=r.get("message",""), trace_id=r.get("trace_id"),
                signals=list(r.get("signals", [])), weight=float(r.get("signal_weight", 0.0))
            ).model_dump())
        clusters.append(EvidenceCluster(
            cluster_id=f"{source_file}::{cat}",
            source_file=source_file,
            candidate_category=cat,
            evidence_count=int(len(g)),
            weighted_score=float(g["signal_weight"].sum()),
            error_count=int((g["level"] == "ERROR").sum()),
            warn_count=int((g["level"] == "WARN").sum()),
            affected_services=services,
            first_seen=min([x for x in g["ts"].dropna().astype(str)] or [None]),
            last_seen=max([x for x in g["ts"].dropna().astype(str)] or [None]),
            evidence_samples=samples
        ).model_dump())
    return sorted(clusters, key=lambda x: x["weighted_score"], reverse=True)

clusters = build_clusters(logs_df)
print("Evidence clusters:", len(clusters))
display(pd.DataFrame([{k:v for k,v in c.items() if k!="evidence_samples"} for c in clusters]).head(20))

## RAG-Ready Runbook Library

In [ ]:
RUNBOOKS = [
    {"title":"Database Connection Pool Exhaustion","category":"database","content":"HikariPool timeout, remaining connection slots, active sessions, lock waits, slow queries. Check pg_stat_activity and blockers. Tune pool only after DB capacity check. Roll back query/deploy if correlated."},
    {"title":"Network Instability","category":"network","content":"DNS failure, TLS handshake timeout, connection reset, upstream connect failure. Check ingress, DNS, network policy, cert chain, load balancer health."},
    {"title":"Authentication Failures","category":"authentication","content":"JWT issuer mismatch, invalid signature, JWK refresh failure, 401/403 spike, RBAC deny. Validate issuer/audience/kid and rollback config if needed."},
    {"title":"Memory CPU Pressure","category":"memory_cpu","content":"OOMKilled, heap pressure, GC pause, CPU throttling. Inspect limits, heap, restart count, autoscaling, recent releases."},
    {"title":"Deployment Regression","category":"deployment_regression","content":"Rollout, new version, feature flag, configmap, canary failure, new error signature. Compare before/after and rollback or disable flag."},
    {"title":"API Timeout","category":"api_timeout","content":"HTTP 504, gateway timeout, p95 latency, circuit breaker. Identify slow upstream, check saturation, use fallback. Avoid blindly increasing timeout."},
    {"title":"Queue Backlog","category":"queue_backlog","content":"Consumer lag, DLQ, retry storm, poison message, rebalance loop. Inspect partitions, lag, payload schema, consumers."},
    {"title":"Disk Storage","category":"disk_storage","content":"No space left, disk pressure, logrotate failure, iowait, PV latency. Clean logs, fix rotation, expand volume, move workload."},
    {"title":"External Dependency","category":"external_dependency","content":"Provider 5xx, vendor latency, 429, Retry-After, feed delay. Check status page, fallback provider, backoff, stakeholder communication."},
    {"title":"Unknown Investigation","category":"unknown","content":"Missing correlation ID, incomplete trace, unknown state, redacted exception. Collect more telemetry and create investigation ticket."}
]

def retrieve_runbooks(category, evidence_text, k=2):
    query_tokens = set(re.findall(r"[a-zA-Z0-9_]{4,}", (category + " " + evidence_text).lower()))
    scored = []
    for rb in RUNBOOKS:
        text = (rb["title"] + " " + rb["category"] + " " + rb["content"]).lower()
        score = sum(1 for t in query_tokens if t in text)
        if rb["category"] == category:
            score += 10
        scored.append((score, rb))
    return [rb for score, rb in sorted(scored, key=lambda x: x[0], reverse=True)[:k]]

## LLM Helper and LangGraph Agents

In [ ]:
def safe_json_loads(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
    return None

def llm_json(system_prompt, user_prompt, fallback_fn, temperature=0.1):
    if not USE_LLM:
        return fallback_fn()
    try:
        llm = get_llm(temperature=temperature)
        resp = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
        data = safe_json_loads(resp.content)
        return data if data is not None else fallback_fn()
    except Exception as e:
        print("LLM failed; fallback used:", repr(e))
        return fallback_fn()

def evidence_text(cluster, max_items=10):
    return "\n".join([
        f"{ev.get('source_file')}:{ev.get('line_no')} [{ev.get('level')}] {ev.get('service')} :: {ev.get('message')}"
        for ev in cluster.get("evidence_samples", [])[:max_items]
    ])

def fallback_severity(cluster):
    cat, score, err = cluster["candidate_category"], cluster["weighted_score"], cluster["error_count"]
    p1_cats = {"database", "memory_cpu", "disk_storage", "deployment_regression"}
    if score >= 120 or err >= 35:
        return "P1" if cat in p1_cats else "P2"
    if score >= 35 or err >= 8:
        return "P2"
    return "P3"

class GraphState(TypedDict):
    current_cluster: dict
    classification: dict
    timeline: list[dict]
    rca: dict
    remediation: dict
    critic: dict
    incident: dict
    incidents: list[dict]
    errors: list[str]

def classifier_agent_node(state: GraphState):
    cluster = state["current_cluster"]
    evtext = evidence_text(cluster)
    runbooks = retrieve_runbooks(cluster["candidate_category"], evtext)
    system = "You are a senior SRE incident classifier. Return strict JSON only."
    user = f"""
Infer category and severity from raw evidence.

Candidate category: {cluster['candidate_category']}
Evidence count: {cluster['evidence_count']}
Error count: {cluster['error_count']}
Weighted score: {cluster['weighted_score']}
Affected services: {cluster['affected_services']}

Evidence:
{evtext}

Runbook hints:
{json.dumps(runbooks, indent=2)}

Return JSON with keys:
category, severity, confidence, reasoning, symptom_vs_cause.
"""
    def fallback():
        conf = min(0.98, 0.35 + 0.25*min(1, cluster["evidence_count"]/50) + 0.40*min(1, cluster["weighted_score"]/160))
        return {
            "category": cluster["candidate_category"],
            "severity": fallback_severity(cluster),
            "confidence": round(conf, 2),
            "reasoning": f"Repeated {cluster['candidate_category']} signals found in {cluster['evidence_count']} evidence lines.",
            "symptom_vs_cause": "Fallback mode cannot deeply distinguish symptom from root cause."
        }
    state["classification"] = llm_json(system, user, fallback)
    return state

def timeline_agent_node(state: GraphState):
    cluster = state["current_cluster"]
    samples = sorted(cluster.get("evidence_samples", []), key=lambda x: (str(x.get("ts")), x.get("line_no", 0)))
    state["timeline"] = [{
        "ts": ev.get("ts"), "level": ev.get("level"), "service": ev.get("service"),
        "line": ev.get("line_no"), "message": ev.get("message"), "trace_id": ev.get("trace_id")
    } for ev in samples[:20]]
    return state

def rca_agent_node(state: GraphState):
    cluster, classification = state["current_cluster"], state["classification"]
    evtext = evidence_text(cluster)
    runbooks = retrieve_runbooks(classification["category"], evtext, k=3)
    system = "You are a principal SRE performing evidence-grounded RCA. Return strict JSON only."
    user = f"""
Use only the evidence and runbooks.

Classification:
{json.dumps(classification, indent=2)}

Timeline:
{json.dumps(state['timeline'], indent=2)}

Evidence:
{evtext}

Runbooks:
{json.dumps(runbooks, indent=2)}

Return JSON with keys:
primary_root_cause, alternative_causes, supporting_evidence, missing_evidence, confidence.
"""
    def fallback():
        library = {
            "database":"Likely database connection, locking, or query contention causing downstream failures.",
            "network":"Likely DNS, TLS, or upstream connectivity instability.",
            "authentication":"Likely token validation, JWK, issuer, or RBAC issue.",
            "memory_cpu":"Likely CPU or memory pressure causing latency or restarts.",
            "deployment_regression":"Likely deployment, config, or feature flag regression.",
            "api_timeout":"Likely downstream latency causing API or gateway timeouts.",
            "queue_backlog":"Likely consumer lag, poison message, or rebalance instability.",
            "disk_storage":"Likely disk exhaustion, storage pressure, or IO latency.",
            "external_dependency":"Likely third-party provider degradation or throttling.",
            "unknown":"Insufficient evidence for deterministic root cause."
        }
        cat = classification["category"]
        return {
            "primary_root_cause": library.get(cat, library["unknown"]),
            "alternative_causes": ["Correlated downstream symptom", "Recent deployment/config change"],
            "supporting_evidence": [f"{ev.get('source_file')}:{ev.get('line_no')} - {ev.get('message')}" for ev in cluster.get("evidence_samples", [])[:3]],
            "missing_evidence": ["Metrics dashboard", "Deployment events", "Distributed traces"],
            "confidence": classification.get("confidence", 0.5)
        }
    state["rca"] = llm_json(system, user, fallback)
    return state

def remediation_agent_node(state: GraphState):
    c, rca, cluster = state["classification"], state["rca"], state["current_cluster"]
    runbooks = retrieve_runbooks(c["category"], evidence_text(cluster), k=3)
    system = "You are an SRE remediation planner. Return strict JSON only. Prefer safe reversible actions."
    user = f"""
Create remediation plan.

Category: {c['category']}
Severity: {c['severity']}
RCA: {json.dumps(rca, indent=2)}
Runbooks: {json.dumps(runbooks, indent=2)}

Return JSON with keys:
immediate_actions, validation_checks, safety_notes, rollback_or_containment.
"""
    def fallback():
        return {
            "immediate_actions": [
                "Confirm impact and affected services using cited evidence lines.",
                "Check service metrics, saturation, and recent deployment/config events.",
                "Apply relevant runbook steps after human approval."
            ],
            "validation_checks": [
                "Error rate returns to baseline.",
                "Latency and saturation normalize.",
                "No new critical alerts appear after mitigation."
            ],
            "safety_notes": [
                "Human approval required before production changes.",
                "Avoid destructive remediation without service owner confirmation."
            ],
            "rollback_or_containment": [
                "Rollback recent deployment/config if timeline correlation is strong.",
                "Use degraded mode or fallback path if available."
            ]
        }
    state["remediation"] = llm_json(system, user, fallback)
    return state

def critic_agent_node(state: GraphState):
    c, rca, rem, cluster = state["classification"], state["rca"], state["remediation"], state["current_cluster"]
    system = "You are a skeptical incident review critic. Return strict JSON only."
    user = f"""
Review for evidence grounding, hallucination risk, and remediation safety.

Classification: {json.dumps(c)}
RCA: {json.dumps(rca)}
Remediation: {json.dumps(rem)}
Evidence:
{evidence_text(cluster)}

Return JSON with keys:
approved, findings, hallucination_risk, safety_risk, recommended_next_step.
"""
    def fallback():
        findings = []
        if cluster["evidence_count"] < 5:
            findings.append("Low evidence count; RCA is tentative.")
        if c["confidence"] < 0.55:
            findings.append("Low confidence; request more telemetry.")
        if c["severity"] in ["P1", "P2"]:
            findings.append("Human approval required before notification/ticket/remediation.")
        return {
            "approved": True,
            "findings": findings or ["Evidence appears sufficient for a preliminary incident candidate."],
            "hallucination_risk": "medium" if c["confidence"] < 0.7 else "low",
            "safety_risk": "medium" if c["severity"] in ["P1", "P2"] else "low",
            "recommended_next_step": "Proceed with preview actions; require human approval for external changes."
        }
    state["critic"] = llm_json(system, user, fallback)
    return state

def action_builder_node(state: GraphState):
    cluster, c, rca, rem, critic = state["current_cluster"], state["classification"], state["rca"], state["remediation"], state["critic"]
    severity, category = c["severity"], c["category"]
    evidence = cluster.get("evidence_samples", [])[:8]

    top_action = (rem.get("immediate_actions") or ["Review evidence"])[0]
    slack_preview = (
        f"Incident Candidate: {category.replace('_',' ').title()}\\n"
        f"Severity: {severity}\\n"
        f"Confidence: {c.get('confidence')}\\n"
        f"Affected services: {', '.join(cluster.get('affected_services', []))}\\n"
        f"Likely cause: {rca.get('primary_root_cause')}\\n"
        f"Top action: {top_action}\\n"
        f"Status: Preview only; human approval required."
    )

    ticket_preview = {
        "title": f"{severity}: {category.replace('_',' ').title()} incident candidate",
        "labels": ["ai-generated", "incident", severity, category],
        "priority": severity,
        "affected_services": cluster.get("affected_services", []),
        "description": {
            "root_cause": rca.get("primary_root_cause"),
            "alternatives": rca.get("alternative_causes", []),
            "evidence": evidence,
            "remediation": rem.get("immediate_actions", []),
            "validation": rem.get("validation_checks", []),
            "critic": critic
        }
    }

    cookbook = "# Runbook: " + category.replace("_"," ").title() + "\\n\\n"
    cookbook += "## RCA\\n" + str(rca.get("primary_root_cause")) + "\\n\\n"
    cookbook += "## Immediate Actions\\n" + "\\n".join([f"- {x}" for x in rem.get("immediate_actions", [])]) + "\\n\\n"
    cookbook += "## Validation\\n" + "\\n".join([f"- {x}" for x in rem.get("validation_checks", [])]) + "\\n\\n"
    cookbook += "## Safety Notes\\n" + "\\n".join([f"- {x}" for x in rem.get("safety_notes", [])])

    incident = IncidentResult(
        incident_id=cluster["cluster_id"],
        source_file=cluster["source_file"],
        category=category,
        severity=severity,
        confidence=float(c.get("confidence", 0.5)),
        affected_services=cluster.get("affected_services", []),
        first_seen=cluster.get("first_seen"),
        last_seen=cluster.get("last_seen"),
        evidence=evidence,
        timeline=state.get("timeline", []),
        primary_root_cause=rca.get("primary_root_cause", ""),
        alternative_causes=rca.get("alternative_causes", []),
        remediation=rem.get("immediate_actions", []),
        validation_checks=rem.get("validation_checks", []),
        safety_notes=rem.get("safety_notes", []),
        critic_findings=critic.get("findings", []),
        ticket_required=severity in ["P1", "P2"],
        notification_required=severity in ["P1", "P2"],
        slack_preview=slack_preview,
        ticket_preview=ticket_preview,
        cookbook=cookbook
    ).model_dump()

    state["incident"] = incident
    state["incidents"] = state.get("incidents", []) + [incident]
    return state

## Compile LangGraph

In [ ]:
cluster_graph = StateGraph(GraphState)
cluster_graph.add_node("classifier_agent", classifier_agent_node)
cluster_graph.add_node("timeline_agent", timeline_agent_node)
cluster_graph.add_node("rca_agent", rca_agent_node)
cluster_graph.add_node("remediation_agent", remediation_agent_node)
cluster_graph.add_node("critic_agent", critic_agent_node)
cluster_graph.add_node("action_builder", action_builder_node)

cluster_graph.set_entry_point("classifier_agent")
cluster_graph.add_edge("classifier_agent", "timeline_agent")
cluster_graph.add_edge("timeline_agent", "rca_agent")
cluster_graph.add_edge("rca_agent", "remediation_agent")
cluster_graph.add_edge("remediation_agent", "critic_agent")
cluster_graph.add_edge("critic_agent", "action_builder")
cluster_graph.add_edge("action_builder", END)

compiled_cluster_graph = cluster_graph.compile()
print("LangGraph compiled.")

## Run Pipeline

In [ ]:
MAX_CLUSTERS = 30
selected_clusters = clusters[:MAX_CLUSTERS]
all_incidents = []

for cluster in tqdm(selected_clusters, desc="Running LangGraph agents"):
    init_state = {
        "current_cluster": cluster,
        "classification": {},
        "timeline": [],
        "rca": {},
        "remediation": {},
        "critic": {},
        "incident": {},
        "incidents": [],
        "errors": []
    }
    result = compiled_cluster_graph.invoke(init_state)
    all_incidents.extend(result.get("incidents", []))

severity_rank = {"P1": 0, "P2": 1, "P3": 2}
def dedupe_incidents(incidents):
    seen = {}
    for inc in incidents:
        key = (inc["source_file"], inc["category"])
        if key not in seen:
            seen[key] = inc
        else:
            old = seen[key]
            if (severity_rank.get(inc["severity"], 99), -inc["confidence"]) < (severity_rank.get(old["severity"], 99), -old["confidence"]):
                seen[key] = inc
    out = list(seen.values())
    out.sort(key=lambda x: (severity_rank.get(x["severity"], 99), -x["confidence"]))
    return out

final_incidents = dedupe_incidents(all_incidents)
final_df = pd.DataFrame(final_incidents)
print("Final incident candidates:", len(final_df))
display(final_df[["source_file","category","severity","confidence","affected_services","primary_root_cause"]])

## Build Reports and Downloads

In [ ]:
def build_report(incidents):
    lines = []
    lines.append("# Multi-Agent DevOps Incident Analysis Report")
    lines.append("")
    lines.append(f"Generated at: {datetime.utcnow().isoformat()}Z")
    lines.append(f"LLM mode: {USE_LLM}")
    lines.append(f"Model: {MODEL_NAME if USE_LLM else 'fallback deterministic mode'}")
    lines.append("")
    lines.append("## Executive Summary")
    lines.append(f"- Incident candidates: {len(incidents)}")
    lines.append(f"- Severity distribution: `{dict(Counter([i['severity'] for i in incidents]))}`")
    lines.append(f"- Category distribution: `{dict(Counter([i['category'] for i in incidents]))}`")
    lines.append("")
    for idx, inc in enumerate(incidents, start=1):
        lines.append(f"## {idx}. {inc['severity']} - {inc['category'].replace('_',' ').title()}")
        lines.append(f"- Source file: `{inc['source_file']}`")
        lines.append(f"- Confidence: `{inc['confidence']}`")
        lines.append(f"- Affected services: `{', '.join(inc['affected_services'])}`")
        lines.append(f"- First seen: `{inc['first_seen']}`")
        lines.append(f"- Last seen: `{inc['last_seen']}`")
        lines.append(f"- Ticket required: `{inc['ticket_required']}`")
        lines.append(f"- Notification required: `{inc['notification_required']}`")
        lines.append("")
        lines.append("### Root Cause")
        lines.append(inc["primary_root_cause"])
        lines.append("")
        lines.append("### Evidence")
        for ev in inc.get("evidence", [])[:8]:
            lines.append(f"- `{ev.get('source_file')}:{ev.get('line_no')}` [{ev.get('level')}] `{ev.get('service')}` - {ev.get('message')}")
        lines.append("")
        lines.append("### Remediation")
        for x in inc.get("remediation", []):
            lines.append(f"- {x}")
        lines.append("")
        lines.append("### Validation")
        for x in inc.get("validation_checks", []):
            lines.append(f"- {x}")
        lines.append("")
        lines.append("### Critic Findings")
        for x in inc.get("critic_findings", []):
            lines.append(f"- {x}")
        lines.append("")
        lines.append("### Slack Preview")
        lines.append("```")
        lines.append(inc.get("slack_preview", ""))
        lines.append("```")
        lines.append("")
    return "\\n".join(lines)

report_md = build_report(final_incidents)

report_path = OUTPUT_DIR / "incident_report_langgraph.md"
json_path = OUTPUT_DIR / "incident_results_langgraph.json"
csv_path = OUTPUT_DIR / "incident_results_langgraph.csv"
cookbook_path = OUTPUT_DIR / "incident_cookbook_langgraph.md"

report_path.write_text(report_md, encoding="utf-8")
json_path.write_text(json.dumps(final_incidents, indent=2), encoding="utf-8")
pd.DataFrame(final_incidents).to_csv(csv_path, index=False)
cookbook_path.write_text("\\n\\n---\\n\\n".join([i["cookbook"] for i in final_incidents]), encoding="utf-8")

print("Wrote outputs:")
print(report_path)
print(json_path)
print(csv_path)
print(cookbook_path)
print(report_md[:2500])

## Eval and Scoring

In [ ]:
gt_files = list(EXTRACT_DIR.rglob("ground_truth_eval_only/golden_labels.json"))

def normalize_cat(x):
    return str(x).lower().replace("/", "_").replace(" ", "_").strip()

def load_ground_truth(gt_files):
    records = []
    for gt in gt_files:
        data = json.loads(gt.read_text(encoding="utf-8"))
        for item in data:
            sev = item.get("expected_severity")
            records.append({
                "case_id": item.get("case_id"),
                "raw_file": item.get("raw_file"),
                "expected_category": normalize_cat(item.get("expected_category")),
                "expected_severity": sev,
                "expected_ticket_required": sev in ["P1", "P2"],
                "affected_services": item.get("affected_services", []),
            })
    return pd.DataFrame(records)

if not gt_files:
    print("No ground truth found. Eval skipped.")
else:
    gt_df = load_ground_truth(gt_files)
    pred_df = pd.DataFrame(final_incidents)
    pred_df["source_file_basename"] = pred_df["source_file"].apply(lambda x: str(x).split("/")[-1])
    pred_df["category_norm"] = pred_df["category"].apply(normalize_cat)

    rows = []
    for _, gt in gt_df.iterrows():
        raw_base = str(gt["raw_file"]).split("/")[-1]
        candidates = pred_df[(pred_df["source_file_basename"] == raw_base) & (pred_df["category_norm"] == gt["expected_category"])]
        found = len(candidates) > 0
        best = candidates.sort_values("confidence", ascending=False).iloc[0].to_dict() if found else None
        rows.append({
            "case_id": gt["case_id"],
            "raw_file": gt["raw_file"],
            "expected_category": gt["expected_category"],
            "expected_severity": gt["expected_severity"],
            "category_detected": found,
            "predicted_severity": best.get("severity") if best else None,
            "severity_match": (best.get("severity") == gt["expected_severity"]) if best else False,
            "expected_ticket_required": gt["expected_ticket_required"],
            "predicted_ticket_required": best.get("ticket_required") if best else False,
            "ticket_match": (best.get("ticket_required") == gt["expected_ticket_required"]) if best else False,
            "confidence": best.get("confidence") if best else None,
        })

    score_df = pd.DataFrame(rows)
    metrics = {
        "category_recall": float(score_df["category_detected"].mean()),
        "severity_accuracy_on_detected": float(score_df[score_df["category_detected"]]["severity_match"].mean()) if score_df["category_detected"].any() else 0.0,
        "ticket_trigger_accuracy": float(score_df["ticket_match"].mean())
    }

    display(score_df)
    print(json.dumps(metrics, indent=2))

    score_df.to_csv(OUTPUT_DIR / "eval_scores_langgraph.csv", index=False)
    (OUTPUT_DIR / "eval_metrics_langgraph.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

In [ ]:
from google.colab import files

zip_out = OUTPUT_DIR / "langgraph_incident_analysis_outputs.zip"
if zip_out.exists():
    zip_out.unlink()

with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.glob("*"):
        if p.is_file() and p.name != zip_out.name:
            z.write(p, p.name)

print("Output package:", zip_out)
files.download(str(zip_out))

# How to Extend After This Notebook

## Better RAG
Replace the small `RUNBOOKS` list with FAISS/Chroma/Azure AI Search over:
- internal runbooks
- previous postmortems
- service ownership docs
- deployment guides
- error catalogs

## Better scoring
Add:
- evidence-grounding score
- remediation keyword score
- hallucination risk score
- LLM-as-judge qualitative eval
- human reviewer score

## App conversion
Move notebook code into:

```text
app/
  graph/
  core/
  rag/
  integrations/
  evals/
```

Keep Python for deterministic evidence handling. Use LLM/LangGraph for reasoning, RCA, remediation, and critique.